# Credit Agreement Analyzer - Phase 1 Validation
Testing document processing pipeline against a real credit agreement.

In [1]:
from collections import Counter
from pathlib import Path

from credit_analyzer.processing.chunker import Chunker
from credit_analyzer.processing.definitions import DefinitionsIndex, DefinitionsParser
from credit_analyzer.processing.pdf_extractor import PDFExtractor
from credit_analyzer.processing.section_detector import SectionDetector

PDF_PATH = Path(r"../credit_agreement.pdf")

## Step 1: PDF Extraction

In [2]:
extractor = PDFExtractor()
doc = extractor.extract(PDF_PATH)

print(f"Pages: {doc.total_pages}")
print(f"Extraction method: {doc.extraction_method}")
print(f"OCR pages: {sum(1 for p in doc.pages if p.is_ocr)}")
print(f"Pages with tables: {sum(1 for p in doc.pages if p.tables)}")
print(f"\nPage 1 text (first 300 chars):\n{doc.pages[0].text[:300]}")

Pages: 289
Extraction method: mixed
OCR pages: 1
Pages with tables: 52

Page 1 text (first 300 chars):
Exhibit 10.1
Execution Version
SENIOR SECURED CREDIT FACILITIES
CREDIT AGREEMENT
dated as of June 21, 2024,
among
RIBBON COMMUNICATIONS INC.,
as a Guarantor,
RIBBON COMMUNICATIONS OPERATING COMPANY, INC.,
as the Borrower,
THE SEVERAL LENDERS FROM TIME TO TIME PARTY HERETO,
HPS INVESTMENT PARTNERS, L


## Step 2: Section Detection

In [3]:
detector = SectionDetector()
sections = detector.detect_sections(doc)

print(f"Sections found: {len(sections)}")
print(f"Section types: {dict(Counter(s.section_type for s in sections))}")
print()
for s in sections:
    print(f"  {s.section_id:12s} | {s.section_type:25s} | {s.section_title[:50]}")

Sections found: 156
Section types: {'definitions': 9, 'facility_terms': 45, 'representations': 24, 'conditions': 3, 'affirmative_covenants': 15, 'negative_covenants': 19, 'events_of_default': 3, 'agents': 15, 'miscellaneous': 23}

  1.1          | definitions               | Defined Terms
  1.2          | definitions               | Other Definitional Provisions
  1.3          | definitions               | Rounding
  1.4          | definitions               | Currency
  1.5          | definitions               | Limited Condition Acquisitions
  1.6          | definitions               | Rates
  1.7          | definitions               | Agreed Security Principles
  1.8          | definitions               | Irish Terms In this Agreement, where it relates to
  1.9          | definitions               | Dutch Terms Where it relates to a Person incorpora
  2.1          | facility_terms            | Term Commitments
  2.2          | facility_terms            | Procedure for Term Loan Borro

## Step 3: Definitions Parsing

In [4]:
defn_sections = [s for s in sections if s.section_type == "definitions"]
parser = DefinitionsParser()

if defn_sections:
    index = parser.parse(defn_sections[0])
    print(f"Defined terms found: {len(index.definitions)}")
    print()
    # Spot-check key terms
    spot_check = ["ABR", "Borrower", "EBITDA", "Consolidated Net Income",
                  "Available Amount", "Restricted Payment", "Permitted Acquisition"]
    for term in spot_check:
        defn = index.lookup(term)
        if defn:
            print(f"  {term}: {defn[:100]}...")
        else:
            print(f"  {term}: NOT FOUND")
else:
    index = DefinitionsIndex(definitions={})
    print("WARNING: No definitions section found!")

Defined terms found: 391

  ABR: “ABR”: for any day, a rate per annum equal to the highest of (a) the Prime Rate in effect on such da...
  Borrower: “Borrower”: is defined in the preamble hereto....
  EBITDA: NOT FOUND
  Consolidated Net Income: “Consolidated Net Income”: for any period, the consolidated net income (or loss) of Holdings and its...
  Available Amount: “Available Amount”: as at any date of determination, a cumulative amount equal to, without duplicati...
  Restricted Payment: NOT FOUND
  Permitted Acquisition: “Permitted Acquisition”: is defined in Section 7.8(n)....


In [5]:
# Test term detection in a sample sentence
test_text = "The Borrower may make Restricted Payments not to exceed the Available Amount"
found = index.find_terms_in_text(test_text)
print(f"Terms found in sample text: {found}")

Terms found in sample text: ['Restricted Payments', 'Available Amount', 'Borrower']


## Step 4: Chunking

In [6]:
chunker = Chunker()
chunks = chunker.chunk_document(sections, index)

print(f"Total chunks: {len(chunks)}")
print(f"Chunk types: {dict(Counter(c.chunk_type for c in chunks))}")
print(f"Section types: {dict(Counter(c.section_type for c in chunks))}")
print(f"Avg tokens: {sum(c.token_count for c in chunks) / max(len(chunks), 1):.0f}")
print(f"Max tokens: {max((c.token_count for c in chunks), default=0)}")
print(f"Chunks over 800 tokens: {sum(1 for c in chunks if c.token_count > 800)}")

Total chunks: 2978
Chunk types: {'definition': 2565, 'text': 353, 'table': 60}
Section types: {'definitions': 2565, 'facility_terms': 89, 'representations': 25, 'conditions': 6, 'affirmative_covenants': 23, 'negative_covenants': 38, 'events_of_default': 9, 'agents': 21, 'miscellaneous': 202}
Avg tokens: 173
Max tokens: 1445
Chunks over 800 tokens: 58


In [7]:
# Inspect a negative covenant chunk
nc = [c for c in chunks if c.section_type == "negative_covenants"]
if nc:
    sample = nc[0]
    print(f"Section: {sample.section_id} - {sample.section_title}")
    print(f"Type: {sample.chunk_type}, Tokens: {sample.token_count}")
    print(f"Defined terms: {sample.defined_terms_present}")
    print(f"\nText (first 500 chars):\n{sample.text[:500]}")
else:
    print("No negative covenant chunks found")

Section: 7.1 - Financial Condition Covenants
Type: table, Tokens: 144
Defined terms: ['Consolidated Net Leverage Ratio']

Text (first 500 chars):
| Four Consecutive Fiscal Quarter Ending | Consolidated Net Leverage Ratio |
| --- | --- |
| September30, 2024 | 4.75:1.00 |
| December31, 2024 | 4.75:1.00 |
| March31, 2025 | 4.75:1.00 |
| June30, 2025 | 4.75:1.00 |
| September30, 2025 | 4.75:1.00 |
| December31, 2025 | 4.75:1.00 |
| March31, 2026 and each fiscal quarter thereafter | 4.00:1.00 |


In [8]:
# Inspect the biggest chunk (look for anything suspiciously large)
biggest = max(chunks, key=lambda c: c.token_count)
print(f"Biggest chunk: {biggest.token_count} tokens")
print(f"Section: {biggest.section_id} - {biggest.section_title}")
print(f"Type: {biggest.chunk_type}")
print(f"\nText (first 300 chars):\n{biggest.text[:300]}")

Biggest chunk: 1445 tokens
Section: 1.1 - Defined Terms
Type: definition

Text (first 300 chars):
“Consolidated Adjusted EBITDA”: with respect to Holdings and its consolidated Subsidiaries for any period,
(a)
Consolidated Net Income, plus
(b)
the sum, without duplication, of the amounts for such period, but solely to the extent decreasing Consolidated Net
Income for such period, of
(i)
Consolida
